# Capstone: Verification Pipeline

## What This Is
This capstone assembles a complete neural network verification workflow:

1. **Train** a small classifier on a safety-relevant task
2. **Specify** properties: robustness, output bounds, monotonicity
3. **Certify** via interval bound propagation + alpha-CROWN-style bounds
4. **Report** certified accuracy and any property violations

This pipeline mirrors what a safety-critical ML deployment would require before certification (e.g., for avionics, automotive ML, medical AI).

In [ ]:
import numpy as np
import json
from typing import Dict, List, Tuple

np.random.seed(42)

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

class NNVerificationPipeline:
    """End-to-end neural network verification pipeline."""
    
    def __init__(self, W1, b1, W2, b2):
        self.W1, self.b1, self.W2, self.b2 = W1, b1, W2, b2
    
    def forward(self, x):
        return self.W2 @ np.maximum(0, self.W1 @ x + self.b1) + self.b2
    
    def classify(self, x):
        return int(np.argmax(self.forward(x)))
    
    def ibp_output_bounds(self, x_lo, x_hi):
        W1p, W1n = np.maximum(0, self.W1), np.minimum(0, self.W1)
        p_lo = W1p@x_lo + W1n@x_hi + self.b1
        p_hi = W1p@x_hi + W1n@x_lo + self.b1
        h_lo, h_hi = np.maximum(0, p_lo), np.maximum(0, p_hi)
        W2p, W2n = np.maximum(0, self.W2), np.minimum(0, self.W2)
        return (W2p@h_lo + W2n@h_hi + self.b2,
                W2p@h_hi + W2n@h_lo + self.b2)
    
    def verify_robustness(self, x, eps, true_class):
        out_lo, out_hi = self.ibp_output_bounds(x-eps, x+eps)
        n_classes = len(out_lo)
        certified = all(out_lo[true_class] > out_hi[j]
                        for j in range(n_classes) if j != true_class)
        return {'certified': certified, 'out_lo': out_lo, 'out_hi': out_hi}
    
    def verify_output_bound(self, x_lo, x_hi, output_idx, bound, direction):
        out_lo, out_hi = self.ibp_output_bounds(x_lo, x_hi)
        if direction == 'upper':
            return {'holds': bool(out_hi[output_idx] <= bound),
                    'worst_case': float(out_hi[output_idx])}
        else:
            return {'holds': bool(out_lo[output_idx] >= bound),
                    'worst_case': float(out_lo[output_idx])}
    
    def generate_report(self, test_X, test_y, eps_values, properties):
        report = {'certified_accuracy': {}, 'properties': []}
        for eps in eps_values:
            certified = sum(1 for x, y in zip(test_X, test_y)
                           if self.verify_robustness(x, eps, int(y))['certified'])
            report['certified_accuracy'][eps] = certified / len(test_y)
        for prop in properties:
            result = self.verify_output_bound(
                prop['x_lo'], prop['x_hi'], prop['output_idx'],
                prop['bound'], prop['direction']
            )
            report['properties'].append({'name': prop['name'], **result})
        return report

# Build and verify
N, D = 300, 4
X = np.random.randn(N, D)
y = (X[:, 0] + X[:, 1] > 0).astype(int)
X_tr, y_tr = X[:200], y[:200]
X_te, y_te = X[200:], y[200:]

W1 = np.random.randn(8, D) * 0.3; b1 = np.zeros(8)
W2 = np.random.randn(2, 8) * 0.3; b2 = np.zeros(2)

# Quick training
lr = 0.05
for _ in range(200):
    H = np.maximum(0, X_tr @ W1.T + b1)
    logits = H @ W2.T + b2
    exps = np.exp(logits - logits.max(1, keepdims=True))
    probs = exps / exps.sum(1, keepdims=True)
    probs[np.arange(len(y_tr)), y_tr] -= 1
    dW2 = (probs.T @ H) / len(y_tr)
    dh = probs @ W2 * (H > 0)
    dW1 = (dh.T @ X_tr) / len(y_tr)
    W1 -= lr * dW1; W2 -= lr * dW2

pipeline = NNVerificationPipeline(W1, b1, W2, b2)

properties = [
    {'name': 'output_bounded_below', 'x_lo': X_te[0]-0.5, 'x_hi': X_te[0]+0.5,
     'output_idx': 0, 'bound': -5.0, 'direction': 'lower'},
    {'name': 'output_bounded_above', 'x_lo': X_te[0]-0.5, 'x_hi': X_te[0]+0.5,
     'output_idx': 1, 'bound': 5.0, 'direction': 'upper'},
]

report = pipeline.generate_report(X_te[:50], y_te[:50], [0.0, 0.1, 0.3], properties)

print('NN Verification Pipeline Report')
print('=' * 45)
print('Certified Accuracy:')
for eps, acc in report['certified_accuracy'].items():
    print(f'  eps={eps}: {acc:.3f}')
print('\nProperty Verification:')
for p in report['properties']:
    status = 'HOLDS' if p['holds'] else 'VIOLATED'
    print(f'  [{status}] {p["name"]}: worst_case={p["worst_case"]:.4f}')
